# 实验二：数据解析与描述性统计可视化

## Experimental Setup — 数据特征分析

本 Notebook 基于实验一生成的 `master_index.csv`，对多模态数据集进行全面的描述性统计与可视化，
为论文的 **Experimental Setup** 章节提供严谨的数据描述依据。

### 可视化内容
1. **临床窗口期分析**：纵向数据处理方式（入院→出院→随访30天）
2. **标签定义与分布**：区分院内死亡 vs 30天再入院，展示不同病种的标签分布
3. **模态覆盖度（Modality Overlap）**：证明数据量为真实非合成
4. **人口学特征**：年龄/性别分布
5. **ECG 信号质量验证**：展示真实波形样本
6. **影像分布**：CheXchonet 心脏结构指标分布

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib_venn import venn3
from PIL import Image
import wfdb
import warnings
warnings.filterwarnings('ignore')

# For Chinese font support in matplotlib
plt.rcParams['font.sans-serif'] = ['SimHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

# Paths
BASE = r"e:\平时作业\人工智能\data"
MASTER_CSV = os.path.join(BASE, "master_index.csv")
MIMIC_ROOT = os.path.join(BASE, "mimic-iii-clinical-database-demo-1.4")
IMG_ROOT = os.path.join(BASE, "images", "chexchonet-a-chest-radiograph-dataset-with-gold-standard-echocardiography-labels-1.0.0")

# Load Master Index
df = pd.read_csv(MASTER_CSV)
print(f"Master Index loaded: {len(df)} samples")
print(f"Columns: {list(df.columns)}")
df.head()

## 2.1 临床窗口期分析（Clinical Time Window）

展示纵向数据的时间框架：
- 入院时间 → 出院时间 → 30天观察窗口
- 住院时长分布
- ICU 停留时长分布

In [ ]:
# ==============================
# Clinical Window Analysis
# ==============================
df_adm = pd.read_csv(os.path.join(MIMIC_ROOT, "ADMISSIONS.csv"))
df_adm['admittime'] = pd.to_datetime(df_adm['admittime'], errors='coerce')
df_adm['dischtime'] = pd.to_datetime(df_adm['dischtime'], errors='coerce')
df_adm['hospital_los'] = (df_adm['dischtime'] - df_adm['admittime']).dt.total_seconds() / 86400

# Readmission gap
df_adm = df_adm.sort_values(['subject_id', 'admittime'])
df_adm['next_admit'] = df_adm.groupby('subject_id')['admittime'].shift(-1)
df_adm['readmit_gap'] = (df_adm['next_admit'] - df_adm['dischtime']).dt.total_seconds() / 86400

df_icu = pd.read_csv(os.path.join(MIMIC_ROOT, "ICUSTAYS.csv"))

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Plot 1: Hospital LOS distribution
ax1 = axes[0]
df_adm['hospital_los'].clip(0, 60).hist(bins=30, ax=ax1, color='steelblue', edgecolor='white', alpha=0.8)
ax1.axvline(df_adm['hospital_los'].median(), color='red', ls='--', label=f"Median: {df_adm['hospital_los'].median():.1f}d")
ax1.set_xlabel('Hospital Length of Stay (days)')
ax1.set_ylabel('Count')
ax1.set_title('Hospital LOS Distribution')
ax1.legend()

# Plot 2: ICU LOS distribution
ax2 = axes[1]
df_icu['los'].clip(0, 30).hist(bins=30, ax=ax2, color='coral', edgecolor='white', alpha=0.8)
ax2.axvline(df_icu['los'].median(), color='red', ls='--', label=f"Median: {df_icu['los'].median():.1f}d")
ax2.set_xlabel('ICU Length of Stay (days)')
ax2.set_ylabel('Count')
ax2.set_title('ICU LOS Distribution')
ax2.legend()

# Plot 3: Readmission Gap Timeline
ax3 = axes[2]
valid_gaps = df_adm['readmit_gap'].dropna()
valid_gaps_clipped = valid_gaps.clip(0, 365)
ax3.hist(valid_gaps_clipped, bins=50, color='mediumpurple', edgecolor='white', alpha=0.8)
ax3.axvline(30, color='red', ls='--', lw=2, label='30-day threshold')
ax3.axvspan(0, 30, alpha=0.15, color='red', label=f'Readmission window\n(n={int((valid_gaps <= 30).sum())})')
ax3.set_xlabel('Days to Next Admission')
ax3.set_ylabel('Count')
ax3.set_title('Readmission Gap Distribution\n(Clinical Window: 30 days post-discharge)')
ax3.legend()

plt.suptitle('Clinical Time Window Analysis — MIMIC-III Demo', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(BASE, 'fig_clinical_window.png'), dpi=150, bbox_inches='tight')
plt.show()

## 2.2 标签定义与分布

区分两种不良事件：
- **院内死亡（In-Hospital Mortality）**: `hospital_expire_flag=1`
- **30天再入院（30-Day Readmission）**: 出院后 30 天内再次住院
- **复合不良结局（Adverse Outcome）**: 两者取并集

In [ ]:
# ==============================
# Label Distribution Analysis
# ==============================
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# --- Panel A: MIMIC label breakdown ---
df_mimic = df[df['data_source'] == 'MIMIC-III'].copy()
df_supp = df[df['data_source'] == 'SuppCSV'].copy()

ax1 = axes[0]
label_types = ['Readmission\n(30-day)', 'Mortality\n(In-hospital)', 'Adverse\nOutcome']
if 'label_readmission' in df_mimic.columns and 'label_mortality' in df_mimic.columns:
    readmit_pos = (df_mimic['label_readmission'] == 1).sum()
    mort_pos = (df_mimic['label_mortality'] == 1).sum()
    adverse_pos = (df_mimic['label'] == 1).sum()
    readmit_neg = len(df_mimic) - readmit_pos
    mort_neg = len(df_mimic) - mort_pos
    adverse_neg = len(df_mimic) - adverse_pos
    
    x = np.arange(3)
    w = 0.35
    bars1 = ax1.bar(x - w/2, [readmit_neg, mort_neg, adverse_neg], w, label='Negative (0)', color='#4CAF50', alpha=0.8)
    bars2 = ax1.bar(x + w/2, [readmit_pos, mort_pos, adverse_pos], w, label='Positive (1)', color='#F44336', alpha=0.8)
    ax1.set_xticks(x)
    ax1.set_xticklabels(label_types)
    ax1.set_ylabel('Count')
    ax1.set_title('A. MIMIC-III Label Definitions')
    ax1.legend()
    # Add count annotations
    for bar in bars1:
        ax1.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.3, f'{int(bar.get_height())}', ha='center', va='bottom', fontsize=9)
    for bar in bars2:
        ax1.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.3, f'{int(bar.get_height())}', ha='center', va='bottom', fontsize=9)
else:
    # Fallback if columns missing
    df_mimic['label'].value_counts().plot(kind='bar', ax=ax1, color=['#4CAF50', '#F44336'])
    ax1.set_title('A. MIMIC-III Label Distribution')

# --- Panel B: SuppCSV label distribution ---
ax2 = axes[1]
supp_counts = df_supp['label'].value_counts().sort_index()
colors = ['#4CAF50', '#F44336']
supp_counts.plot(kind='bar', ax=ax2, color=colors[:len(supp_counts)], alpha=0.8)
ax2.set_title('B. Heart Failure CSV\n(DEATH_EVENT)')
ax2.set_xlabel('Label')
ax2.set_ylabel('Count')
ax2.set_xticklabels(['Survived (0)', 'Deceased (1)'], rotation=0)
for i, v in enumerate(supp_counts.values):
    ax2.text(i, v + 1, str(v), ha='center', fontweight='bold')

# --- Panel C: Combined label by data source ---
ax3 = axes[2]
cross = pd.crosstab(df['data_source'], df['label'])
cross.plot(kind='bar', stacked=True, ax=ax3, color=['#4CAF50', '#F44336'], alpha=0.8)
ax3.set_title('C. Combined Label by Source')
ax3.set_xlabel('Data Source')
ax3.set_ylabel('Count')
ax3.legend(['Low Risk (0)', 'High Risk (1)'], loc='upper right')
ax3.set_xticklabels(ax3.get_xticklabels(), rotation=0)

plt.suptitle('Label Definition & Distribution Analysis', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(BASE, 'fig_label_distribution.png'), dpi=150, bbox_inches='tight')
plt.show()

## 2.3 模态覆盖度（Modality Overlap）

**论文关键图表**：展示多少样本同时具备三种模态，
证明实验数据为真实多模态而非合成填充。

In [ ]:
# ==============================
# Modality Overlap Visualization
# ==============================

# Calculate sets
has_text  = set(df[df['has_text'] == True].index)
has_ecg   = set(df[df['has_ecg'] == True].index)
has_image = set(df[df['has_image'] == True].index)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# --- Panel A: Venn Diagram ---
ax1 = axes[0]
try:
    v = venn3(
        [has_text, has_ecg, has_image],
        set_labels=('Text\n(Clinical Notes)', 'ECG\n(Time Series)', 'Image\n(Chest X-Ray)'),
        ax=ax1
    )
    ax1.set_title('A. Modality Overlap (Venn Diagram)', fontsize=12, fontweight='bold')
except Exception as e:
    # Fallback if venn3 fails (e.g. all sets identical)
    triple = len(has_text & has_ecg & has_image)
    ax1.text(0.5, 0.5, f'Full Triple Modality:\n{triple} / {len(df)} samples\n({triple/len(df)*100:.1f}%)',
             ha='center', va='center', fontsize=16, transform=ax1.transAxes,
             bbox=dict(boxstyle='round', facecolor='lightblue', alpha=0.8))
    ax1.set_title('A. Modality Coverage', fontsize=12, fontweight='bold')

# --- Panel B: Stacked Bar Chart ---
ax2 = axes[1]
coverage_data = {
    'Text': [df['has_text'].sum(), len(df) - df['has_text'].sum()],
    'ECG': [df['has_ecg'].sum(), len(df) - df['has_ecg'].sum()],
    'Image': [df['has_image'].sum(), len(df) - df['has_image'].sum()]
}
modalities = list(coverage_data.keys())
available = [coverage_data[m][0] for m in modalities]
missing = [coverage_data[m][1] for m in modalities]

x_pos = np.arange(len(modalities))
bars_avail = ax2.bar(x_pos, available, color='#2196F3', alpha=0.8, label='Available')
bars_miss = ax2.bar(x_pos, missing, bottom=available, color='#BDBDBD', alpha=0.6, label='Missing')

for i, (a, m) in enumerate(zip(available, missing)):
    total = a + m
    pct = a / total * 100 if total > 0 else 0
    ax2.text(i, a / 2, f'{a}\n({pct:.0f}%)', ha='center', va='center', fontweight='bold', color='white', fontsize=11)

ax2.set_xticks(x_pos)
ax2.set_xticklabels(modalities, fontsize=12)
ax2.set_ylabel('Number of Samples')
ax2.set_title('B. Per-Modality Coverage', fontsize=12, fontweight='bold')
ax2.legend(loc='upper right')

# Summary stats
triple_count = len(has_text & has_ecg & has_image)
print(f"Modality Coverage Summary:")
print(f"  Text available:   {len(has_text)} / {len(df)} ({len(has_text)/len(df)*100:.1f}%)")
print(f"  ECG available:    {len(has_ecg)} / {len(df)} ({len(has_ecg)/len(df)*100:.1f}%)")
print(f"  Image available:  {len(has_image)} / {len(df)} ({len(has_image)/len(df)*100:.1f}%)")
print(f"  Full Triple:      {triple_count} / {len(df)} ({triple_count/len(df)*100:.1f}%)")

plt.suptitle('Multimodal Data Coverage Analysis', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(BASE, 'fig_modality_overlap.png'), dpi=150, bbox_inches='tight')
plt.show()

## 2.4 人口学特征分布

In [ ]:
# ==============================
# Demographics
# ==============================
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Age distribution by outcome
ax1 = axes[0]
df_plot = df[df['age'] > 0].copy()
for label_val, color, name in [(0, '#4CAF50', 'Low Risk'), (1, '#F44336', 'High Risk')]:
    subset = df_plot[df_plot['label'] == label_val]['age']
    ax1.hist(subset, bins=20, alpha=0.6, color=color, label=f'{name} (n={len(subset)})', edgecolor='white')
ax1.set_xlabel('Age (years)')
ax1.set_ylabel('Count')
ax1.set_title('A. Age Distribution by Outcome')
ax1.legend()

# Gender distribution
ax2 = axes[1]
gender_label = pd.crosstab(df['gender'], df['label'])
gender_label.plot(kind='bar', stacked=True, ax=ax2, color=['#4CAF50', '#F44336'], alpha=0.8)
ax2.set_title('B. Gender × Outcome')
ax2.set_xlabel('Gender')
ax2.set_ylabel('Count')
ax2.legend(['Low Risk', 'High Risk'])
ax2.set_xticklabels(ax2.get_xticklabels(), rotation=0)

# Data source composition
ax3 = axes[2]
source_counts = df['data_source'].value_counts()
wedges, texts, autotexts = ax3.pie(
    source_counts.values, labels=source_counts.index, autopct='%1.1f%%',
    colors=['#2196F3', '#FF9800'], startangle=90, textprops={'fontsize': 11}
)
ax3.set_title('C. Data Source Composition')

plt.suptitle('Cohort Demographics', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(BASE, 'fig_demographics.png'), dpi=150, bbox_inches='tight')
plt.show()

# Print Table 1 (for paper)
print("\n=== Table 1: Cohort Characteristics ===")
print(f"Total samples: {len(df)}")
print(f"MIMIC-III: {len(df[df['data_source']=='MIMIC-III'])}")
print(f"Supplementary HF CSV: {len(df[df['data_source']=='SuppCSV'])}")
print(f"Age (mean ± std): {df_plot['age'].mean():.1f} ± {df_plot['age'].std():.1f}")
print(f"Female %: {(df['gender'].isin(['F','Female'])).mean()*100:.1f}%")
print(f"Positive label rate: {df['label'].mean()*100:.1f}%")

## 2.5 ECG 信号质量验证

从对齐后的数据集中随机抽取 ECG 样本，展示真实波形。

In [ ]:
# ==============================
# ECG Signal Quality Check
# ==============================
ecg_samples = df[df['has_ecg'] == True].sample(min(6, df['has_ecg'].sum()), random_state=42)

fig, axes = plt.subplots(2, 3, figsize=(18, 8))
axes_flat = axes.flatten()

for i, (_, row) in enumerate(ecg_samples.iterrows()):
    if i >= 6: break
    ax = axes_flat[i]
    try:
        sig, meta = wfdb.rdsamp(row['ecg_path'], channels=[0])
        raw = sig.flatten()
        fs = meta['fs']
        # Plot first 5 seconds or all
        n_pts = min(int(5 * fs), len(raw))
        t = np.arange(n_pts) / fs
        ax.plot(t, raw[:n_pts], color='black', lw=0.6)
        
        source = os.path.basename(os.path.dirname(row['ecg_path']))
        label_str = 'High Risk' if row['label'] == 1 else 'Low Risk'
        ax.set_title(f"{os.path.basename(row['ecg_path'])}\n{label_str} | fs={fs}Hz", fontsize=9)
        ax.set_xlabel('Time (s)')
        ax.set_ylabel('mV')
    except Exception as e:
        ax.text(0.5, 0.5, f'Error: {str(e)[:30]}', ha='center', va='center', transform=ax.transAxes)
        ax.set_title(f'Failed to load', fontsize=9)

plt.suptitle('ECG Signal Quality Verification (Random Samples from Aligned Dataset)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(BASE, 'fig_ecg_samples.png'), dpi=150, bbox_inches='tight')
plt.show()

## 2.6 影像数据分布（CheXchonet 心脏结构指标）

In [ ]:
# ==============================
# Image / Cardiac Structure Analysis
# ==============================
metadata_path = os.path.join(IMG_ROOT, "metadata.csv")
df_img = pd.read_csv(metadata_path)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# LVIDd distribution
ax1 = axes[0]
df_img['lvidd'].dropna().hist(bins=40, ax=ax1, color='steelblue', edgecolor='white', alpha=0.8)
ax1.axvline(5.6, color='red', ls='--', lw=2, label='Dilated threshold (M): 5.6cm')
ax1.axvline(5.2, color='orange', ls='--', lw=2, label='Dilated threshold (F): 5.2cm')
ax1.set_xlabel('LVIDd (cm)')
ax1.set_ylabel('Count')
ax1.set_title('A. LV Internal Diameter (Diastole)')
ax1.legend(fontsize=8)

# IVSd distribution
ax2 = axes[1]
df_img['ivsd'].dropna().hist(bins=40, ax=ax2, color='coral', edgecolor='white', alpha=0.8)
ax2.axvline(1.1, color='red', ls='--', lw=2, label='Hypertrophy threshold: 1.1cm')
ax2.set_xlabel('IVSd (cm)')
ax2.set_ylabel('Count')
ax2.set_title('B. Interventricular Septum (Diastole)')
ax2.legend(fontsize=8)

# Structural abnormality flags
ax3 = axes[2]
struct_counts = pd.DataFrame({
    'Condition': ['SLVH', 'DLV', 'Composite\n(SLVH|DLV)', 'Heart\nTransplant', 'Pacemaker\n/ICD'],
    'Count': [
        df_img['slvh'].sum() if 'slvh' in df_img.columns else 0,
        df_img['dlv'].sum() if 'dlv' in df_img.columns else 0,
        df_img['composite_slvh_dlv'].sum() if 'composite_slvh_dlv' in df_img.columns else 0,
        df_img['heart_transplant'].sum() if 'heart_transplant' in df_img.columns else 0,
        df_img['pacemaker_or_icd'].sum() if 'pacemaker_or_icd' in df_img.columns else 0
    ]
})
sns.barplot(data=struct_counts, x='Condition', y='Count', ax=ax3, palette='Set2')
ax3.set_title('C. Cardiac Structural Abnormalities')
ax3.set_ylabel('Count')
for i, v in enumerate(struct_counts['Count']):
    ax3.text(i, v + 10, str(int(v)), ha='center', fontweight='bold')

plt.suptitle('CheXchonet Image Dataset — Cardiac Structure Analysis', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(BASE, 'fig_image_cardiac.png'), dpi=150, bbox_inches='tight')
plt.show()

print(f"\nCheXchonet Summary:")
print(f"  Total images: {len(df_img)}")
print(f"  LVIDd mean: {df_img['lvidd'].mean():.2f} cm")
print(f"  Structural abnormality rate: {df_img['composite_slvh_dlv'].mean()*100:.1f}%")

## 2.7 综合数据摘要表（论文用）

In [ ]:
# ==============================
# Summary Table for Paper
# ==============================
triple_count = ((df['has_text']) & (df['has_ecg']) & (df['has_image'])).sum()
df_valid_age = df[df['age'] > 0]

summary = {
    'Dataset Metric': [
        'Total Samples',
        'MIMIC-III Admissions',
        'Supplementary HF Records',
        'Text Modality Coverage',
        'ECG Modality Coverage',
        'Image Modality Coverage',
        'Full Triple-Modal Samples',
        'Positive Label Rate (Adverse)',
        'Mean Age (years)',
        'Female Proportion',
        'ECG Sources (Total Files)',
        'Image Source (CheXchonet)'
    ],
    'Value': [
        len(df),
        len(df[df['data_source'] == 'MIMIC-III']),
        len(df[df['data_source'] == 'SuppCSV']),
        f"{df['has_text'].sum()} ({df['has_text'].mean()*100:.1f}%)",
        f"{df['has_ecg'].sum()} ({df['has_ecg'].mean()*100:.1f}%)",
        f"{df['has_image'].sum()} ({df['has_image'].mean()*100:.1f}%)",
        f"{triple_count} ({triple_count/len(df)*100:.1f}%)",
        f"{df['label'].mean()*100:.1f}%",
        f"{df_valid_age['age'].mean():.1f} ± {df_valid_age['age'].std():.1f}",
        f"{(df['gender'].isin(['F','Female'])).mean()*100:.1f}%",
        f"{df['has_ecg'].sum()} (PhysioNet + CPSC + CHF + Files)",
        f"{df['has_image'].sum()} (CheXchonet)"
    ]
}

df_summary = pd.DataFrame(summary)
print("\n" + "="*60)
print("TABLE 1: Dataset Summary for Experimental Setup")
print("="*60)
for _, row in df_summary.iterrows():
    print(f"  {row['Dataset Metric']:40s} {row['Value']}")
print("="*60)

# Also save as CSV for easy LaTeX table generation
df_summary.to_csv(os.path.join(BASE, 'table1_dataset_summary.csv'), index=False)
print(f"\nTable saved to: {os.path.join(BASE, 'table1_dataset_summary.csv')}")